# Fine-tuning MiewID

The public MiewID-msv3 weights, trained on many species, are fine-tuned on the training split
with a sub-centre ArcFace loss.

- **Reads** the 2048 px JPEG crops in `data_root/segmented_compressed/`, as
  `3_extract_embeddings.ipynb` does.  
- **Writes** the checkpoint to `miewid_checkpoint` of `../config.toml`, and
  `training_history.json` next to it after every epoch. 

## Settings

In [4]:
import tomllib
from pathlib import Path

ROOT = Path.cwd().parent
CONFIG = tomllib.loads((ROOT / "config.toml").read_text())

# Relative paths in config.toml start at the folder that holds it.
DATA_ROOT = ROOT / Path(CONFIG["data_root"]).expanduser()
SEGMENTED_COMPRESSED = DATA_ROOT / "segmented_compressed"
assert SEGMENTED_COMPRESSED.exists(), f"Not found: {SEGMENTED_COMPRESSED}. Check {ROOT / 'config.toml'}."

## Train/validation/test split

In [ ]:
import re

import pandas as pd
from itables import init_notebook_mode, show

init_notebook_mode(all_interactive=True)  # every DataFrame shown from here on is interactive

# 'SH024_002_007_det0_cropped.jpg' -> individual 'SH024', observation number '002'. The photo number
# and SAM 3's suffixes (_det<n>, _segmented, _mask, _cropped, _annotated) are optional.
PHOTO_NAME = re.compile(r"^(?P<identity>SH\d+)_(?P<number>\d+)(?:_\d+)?"
                        r"(?:_det\d+|_segmented|_mask|_cropped|_annotated)*(?:\.[^.]+)?$")

# The image table: one row per crop, sorted by name, with its individual and its observation.
names = pd.Series(sorted(path.name for path in SEGMENTED_COMPRESSED.glob("*.jpg")))
parts = names.str.extract(PHOTO_NAME)
skipped = names[parts["identity"].isna()].tolist()
if skipped:
    print(f"Skipped {len(skipped)} files whose names do not follow the "
          f"SH<identity>_<observation>_<photo> convention, e.g. {skipped[:3]}")
crops = (pd.DataFrame({"name": names, "identity": parts["identity"],
                       "observation": parts["identity"] + "_" + parts["number"]})
         .dropna().reset_index(drop=True))
show(crops, scrollY="400px", scroller=True, maxBytes=0, showIndex=False)

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42
TRAIN_FRACTION, VALIDATION_FRACTION = 0.70, 0.10  # the test split takes the remaining 20%

# Every photograph of an individual goes to the same split, so the split is drawn over individuals.
# It is stratified by their number of observations, in bins: 1, 2, 3, 4, 5-9 and 10 or more. A
# stratum needs at least two individuals, and most counts above 10 have only one.
observation_count = crops.groupby("identity")["observation"].nunique()
stratum = pd.Series(np.digitize(observation_count, [2, 3, 4, 5, 10]), index=observation_count.index)
# First 70% for training, then the other 30% divided 1:2 into validation and test.
train_ids, rest = train_test_split(observation_count.index, train_size=TRAIN_FRACTION,
                                   stratify=stratum, random_state=SEED)
validation_ids, test_ids = train_test_split(rest, train_size=VALIDATION_FRACTION / (1 - TRAIN_FRACTION),
                                            stratify=stratum[rest], random_state=SEED)
split_by_identity = {**dict.fromkeys(train_ids, "train"), **dict.fromkeys(validation_ids, "validation"),
            **dict.fromkeys(test_ids, "test")}
images = crops.assign(split=crops["identity"].map(split_by_identity))
show(images, scrollY="400px", scroller=True, maxBytes=0, showIndex=False)

<TableReport: use .open() or .markdown() to display>

In [7]:
train = images[images["split"] == "train"].reset_index(drop=True)
validation = images[images["split"] == "validation"].reset_index(drop=True)

# ArcFace needs class numbers 0..N-1.
# The ArcFace class of each photograph: its individual's number, individuals sorted by name.
codes, individuals = pd.factorize(train["identity"], sort=True)
labels = codes.tolist()  # Python ints, for the sampler and the loss
print(f"training on {len(train)} photographs of {len(individuals)} individuals")

training on 21354 photographs of 331 individuals


## Selecting the checkpoint

The validation individuals are split into five sub-folds, stratified by their number of
observations. In fold *i*, the individuals of sub-fold *i* leave the gallery, and those of
sub-fold *i + 1* with two or more observations are the known queries. The score is the mean
mAP@R over the folds, on known queries only.
The mask removes each query's own observation from its gallery; otherwise every query would find
itself.

mAP@R is the average precision over each query's first R results, R being its number of matches.
It reaches 1 when every match ranks above every non-match, which is what the shortlist of the
rerank needs.

In [ ]:
from sklearn.model_selection import StratifiedKFold

SELECTION_FOLDS = 5

# Five sub-folds of the validation individuals, stratified by their number of observations in bins
# 1, 2-4 and 5 or more: finer bins would leave some sub-folds without an individual of a bin.
observation_count = validation.groupby("identity")["observation"].nunique()
validation_individuals = observation_count.index
folds = StratifiedKFold(n_splits=SELECTION_FOLDS, shuffle=True, random_state=SEED + 1)
sub_folds = [set(validation_individuals[members])
             for _, members in folds.split(validation_individuals, np.digitize(observation_count, [2, 5]))]

# Fold i hides the individuals of sub-fold i from the gallery. The known queries are those of
# sub-fold i + 1 with two or more observations, so another observation stays in the gallery.
can_be_known = set(validation_individuals[observation_count >= 2])
selection_folds = []
for index, hidden in enumerate(sub_folds):
    known = sub_folds[(index + 1) % SELECTION_FOLDS] & can_be_known
    gallery = validation[~validation["identity"].isin(hidden)].reset_index(drop=True)
    queries = validation[validation["identity"].isin(known)].reset_index(drop=True)
    selection_folds.append((gallery, queries))

In [ ]:
pd.DataFrame({"hidden individuals": [", ".join(sorted(hidden)) for hidden in sub_folds],
              "gallery photographs": [len(gallery) for gallery, _ in selection_folds],
              "query photographs": [len(queries) for _, queries in selection_folds]})

## The model and the data

Validation images go through exactly the preprocessing of `3_extract_embeddings.ipynb`, so the
mAP@R that selects the checkpoint sees the same input as the cached embeddings. Training images
are augmented: scale, colour, greyscale and blur.

All BatchNorm layers stay in inference mode. A batch holds a few individuals only, so its
statistics are too noisy to estimate again; the pretrained running statistics are better. MiewID's
own classification head predicts the identities of its first training, other species. The forward
pass does not call it, so it takes no gradient. It stays in the model only so the saved weights
match the model that `3_extract_embeddings.ipynb` builds.

In [ ]:
import importlib
import math
import sys
import types

import torch
import torch.nn as nn
from huggingface_hub import snapshot_download
from PIL import Image, ImageOps
from pytorch_metric_learning import losses as metric_losses
from pytorch_metric_learning import samplers
from safetensors.torch import load_file
from torch.utils.data import DataLoader
from torchvision import transforms

assert torch.cuda.is_available(), "This notebook needs a GPU."
DEVICE = "cuda"
MIEWID_SIZE = 440  # input side in pixels
EPOCHS = 100
IMAGES_PER_IDENTITY = 4
BATCH_SIZE = 24        # 6 individuals x 4 images
BACKBONE_LR = 2e-5
HEAD_LR = 1e-3
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
SUB_CENTERS = 3        # sub-centre ArcFace, as in MiewID's own training
ARCFACE_SCALE = 64.0
ARCFACE_MARGIN = 0.5   # radians
IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
MIEWID_MODEL = "conservationxlabs/miewid-msv3"
MIEWID_DIM = 2152  # length of a MiewID-msv3 embedding

blur_kernel = max(3, round(MIEWID_SIZE * 0.05) | 1)  # odd, about 5% of the image
TRAIN_TRANSFORM = transforms.Compose([
    # Centre the crop on a white square, so no view is stretched. White matches the
    # background that the segmentation paints, exactly (255, 255, 255).
    transforms.Lambda(lambda image: ImageOps.pad(image, (max(image.size),) * 2,
                                                 color=(255, 255, 255))), # converts to a square of size max(image.size) x max(image.size)
    transforms.RandomResizedCrop(MIEWID_SIZE, scale=(0.75, 1.0), ratio=(1.0, 1.0), antialias=True,
                                 interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.2, 0.05)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(blur_kernel, sigma=(0.1, 2.0))], p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def train_batch(items):
    """A training batch from (path, label) pairs: augmented images and their labels."""
    return (torch.stack([TRAIN_TRANSFORM(Image.open(path).convert("RGB")) for path, _ in items]),
            torch.tensor([label for _, label in items]))


def freeze_batchnorm(model):
    """Put every BatchNorm in inference mode and freeze its parameters; return how many."""
    frozen = 0
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            for parameter in module.parameters(recurse=False):
                frozen += parameter.numel()
                parameter.requires_grad = False
    return frozen


torch.manual_seed(SEED)
np.random.seed(SEED)

# MiewID is published on Hugging Face as loose Python files, not as a library. The downloaded
# folder is therefore registered under a made-up package name and imported from there.
model_dir = Path(snapshot_download(MIEWID_MODEL))
package_name = f"_miewid_{model_dir.name.replace('-', '_')}"
if package_name not in sys.modules:
    package = types.ModuleType(package_name)
    package.__path__ = [str(model_dir)]
    sys.modules[package_name] = package
modeling = importlib.import_module(f"{package_name}.modeling_miewid")
configuration = importlib.import_module(f"{package_name}.configuration_miewid")
config = configuration.MiewIdNetConfig.from_pretrained(MIEWID_MODEL)
config.pretrained = False  # the next line loads the weights
model = modeling.MiewIdNet(config)
model.load_state_dict(load_file(str(model_dir / "model.safetensors")), strict=False)
for parameter in model.final.parameters():  # MiewID's own classification head
    parameter.requires_grad = False
frozen = freeze_batchnorm(model)
model.to(DEVICE)
arcface = metric_losses.SubCenterArcFaceLoss(
    num_classes=len(individuals), embedding_size=MIEWID_DIM, margin=math.degrees(ARCFACE_MARGIN),
    scale=ARCFACE_SCALE, sub_centers=SUB_CENTERS).to(DEVICE)
print(f"trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} backbone "
      f"(+{frozen:,} frozen BatchNorm), {sum(p.numel() for p in arcface.parameters()):,} ArcFace")

# The pretrained backbone moves slowly, so its general animal features survive. The ArcFace
# centres start at random and must move fast.
optimizer = torch.optim.AdamW([
    {"params": [p for p in model.parameters() if p.requires_grad], "lr": BACKBONE_LR},
    {"params": list(arcface.parameters()), "lr": HEAD_LR},
], weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
scaler = torch.amp.GradScaler(DEVICE)

# ArcFace compares each image with the centres of the individuals, not with the other images of
# the batch. The sampler gives every individual the same share of the batches, so individuals
# with many photos do not dominate. An individual with fewer than 4 photos repeats some of them.
items = list(zip([SEGMENTED_COMPRESSED / name for name in train["name"]], labels))
loader = DataLoader(items, batch_size=BATCH_SIZE, num_workers=4, drop_last=True, collate_fn=train_batch,
                    sampler=samplers.MPerClassSampler(labels, m=IMAGES_PER_IDENTITY, batch_size=BATCH_SIZE,
                                                      length_before_new_iter=len(items)))

## Training

The checkpoint on disk is always the best one so far, not the last. The history is written
after every epoch, so a run that stops early still leaves a usable curve.

In [ ]:
import json
import time

from tqdm.auto import tqdm

CHECKPOINT = ROOT / Path(CONFIG["miewid_checkpoint"]).expanduser()
EVAL_EVERY = 5         # epochs between two validation passes


# Square-pad, resize with Lanczos, rescale the pixels to 0-1 with colour first, then centre each colour
# on the model's training statistics.
EVAL_TRANSFORM = transforms.Compose([
    # Centre the crop on a white square, so no view is stretched. White matches the
    # background of the segmentation.
    transforms.Lambda(lambda image: ImageOps.pad(image, (max(image.size),) * 2,
                                                 color=(255, 255, 255))),
    transforms.Resize((MIEWID_SIZE, MIEWID_SIZE), interpolation=transforms.InterpolationMode.LANCZOS),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def eval_batch(paths):
    """An evaluation batch: the images preprocessed as for the embedding caches."""
    return torch.stack([EVAL_TRANSFORM(Image.open(path).convert("RGB")) for path in paths])


@torch.no_grad()
def embed(model, names):
    """Unit-length MiewID embeddings of some crops."""
    model.eval()
    loader = DataLoader([SEGMENTED_COMPRESSED / name for name in names], batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, collate_fn=eval_batch)
    vectors = [nn.functional.normalize(model(batch.to(DEVICE)), p=2, dim=1).cpu().numpy() for batch in loader]
    return np.concatenate(vectors)  # float32, as the model


CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
history, best = [], -np.inf
for epoch in range(1, EPOCHS + 1):
    model.train() # sets the training flag to True on the model and all its submodules.
    freeze_batchnorm(model)
    started, total = time.time(), 0.0
    for batch_images, batch_labels in tqdm(loader, desc=f"epoch {epoch}/{EPOCHS}", leave=False):
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(DEVICE):
            loss = arcface(model(batch_images.to(DEVICE)), batch_labels.to(DEVICE)) # run the forward pass in mixed precision to save time and memory.
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += float(loss.detach())
    scheduler.step()

    entry = {"epoch": epoch, "loss": total / max(1, len(loader)), "seconds": round(time.time() - started)}
    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        # Mean mAP@R over the selection folds, with the rich gallery. Masked gallery entries get
        # -inf, so they are never ranked.
        fold_scores = []
        for gallery, query in selection_folds:
            mask = query["observation"].to_numpy()[:, None] != gallery["observation"].to_numpy()[None, :]
            query_identities = query["identity"].to_numpy()
            gallery_identities = gallery["identity"].to_numpy()
            similarity = np.where(mask, embed(model, query["name"].tolist()) @ embed(model, gallery["name"].tolist()).T,
                                  -np.inf)
            precisions = []  # the average precision at R of each query
            for row in range(similarity.shape[0]):
                relevant = (gallery_identities == query_identities[row]) & mask[row]
                n_relevant = relevant.sum()
                if n_relevant == 0:
                    continue
                hits = relevant[np.argsort(-similarity[row])[:n_relevant]]
                precision = np.cumsum(hits) / np.arange(1, n_relevant + 1)
                precisions.append((precision * hits).sum() / n_relevant)
            fold_scores.append(np.mean(precisions) if precisions else 0.0)
        entry["mAP@R"] = float(np.mean(fold_scores))
        if entry["mAP@R"] > best:
            best = entry["mAP@R"]
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "mAP@R": best}, CHECKPOINT)
            entry["saved"] = True
    history.append(entry)
    print(json.dumps(entry))
    (CHECKPOINT.parent / "training_history.json").write_text(json.dumps(history, indent=1))
print(f"best mAP@R {best:.4f} -> {CHECKPOINT}")